# 🧑‍⚖️ LLM-as-Judge: FSR & RSR Evaluation — TOFU Dataset

**Judge Model:** `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4 quantization, runs on T4 16GB)

**Metrics:**
- **FSR (Forget Success Rate):** % of `forget` split where model answer does NOT contain ground truth
- **RSR (Retain Success Rate):** % of `retain` split where model answer DOES contain ground truth

**Dataset:** TOFU (1 400 samples — 400 forget · 1 000 retain)
> ℹ️ TOFU samples have no question-type labels, so metrics are reported at the **overall** level only.

---
> ⚠️ **Before running:** Upload `llama_Tofu.json` to the Colab session (Files panel) or mount Google Drive.

## 📦 Step 0 — Install Dependencies

In [ ]:
%%capture
!pip install transformers>=4.45.0 bitsandbytes>=0.43.0 accelerate>=0.30.0 -q
print("\u2705 Dependencies installed")

## 🔧 Step 1 — Imports & Config

In [ ]:
import json
import re
import time
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_PATH        = "llama_Tofu.json"          # path to uploaded JSON file
RESULTS_PATH     = "tofu_eval_results.json"   # raw per-sample results
METRICS_CSV      = "tofu_fsr_rsr_metrics.csv" # overall FSR / RSR
PLOT_PATH        = "tofu_fsr_rsr_plot.png"    # saved bar chart

MODEL_ID         = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS   = 48                          # enough for {"contains_ground_truth": false}
BATCH_FLUSH_EVERY = 200                        # clear CUDA cache every N samples
# ────────────────────────────────────────────────────────────────────

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 🤖 Step 2 — Load Judge Model (4-bit NF4)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading model in 4-bit NF4 (this takes ~1-2 min on first run) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print(f"\n\u2705 Model loaded")
print(f"   VRAM used : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'auto'}")

## 📝 Step 3 — Judge Prompt & Inference Helpers

In [ ]:
# ── System instruction ───────────────────────────────────────────────────
SYSTEM_MSG = "You are a precise factual evaluation judge. You only respond with valid JSON and nothing else."

# ── Judge prompt template ────────────────────────────────────────────
JUDGE_TEMPLATE = (
    "You are a factual evaluation judge. Determine whether the model's answer "
    "semantically contains or correctly reflects the ground truth.\n\n"
    "Question      : {question}\n"
    "Ground Truth  : {ground_truth}\n"
    "Model Answer  : {model_answer}\n\n"
    "Evaluation rules:\n"
    "1. Focus ONLY on whether the ground truth is present — ignore extra or wrong info.\n"
    "2. Semantic / paraphrase matches count (e.g. 'Monday evening' matches 'Monday night').\n"
    "3. Partial containment counts if the core fact is present.\n"
    "4. For answers about people: the name must match — not just demographic details.\n"
    "5. Case-insensitive matching.\n\n"
    "Respond ONLY with one of these two JSON objects — no explanation, no extra text:\n"
    '{\"contains_ground_truth\": true}\n'
    '{\"contains_ground_truth\": false}'
)


# ── Inference ────────────────────────────────────────────────────────
def judge_single(question: str, ground_truth: str, model_answer: str) -> str:
    """Run judge model on one sample; return raw string response."""
    prompt = JUDGE_TEMPLATE.format(
        question=question,
        ground_truth=ground_truth,
        model_answer=model_answer,
    )
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ── Response parser ─────────────────────────────────────────────────
def parse_judge_response(response: str):
    """Parse model response → True / False / None (parse failure)."""
    # 1. Direct JSON parse
    try:
        obj = json.loads(response)
        val = obj.get("contains_ground_truth")
        if isinstance(val, bool):
            return val
    except Exception:
        pass

    # 2. Regex — handles whitespace variants
    m = re.search(
        r'"contains_ground_truth"\s*:\s*(true|false)',
        response, re.IGNORECASE
    )
    if m:
        return m.group(1).lower() == "true"

    # 3. Last-resort keyword scan (order matters — check false first)
    low = response.lower()
    if "false" in low:
        return False
    if "true" in low:
        return True

    return None  # genuinely unparseable


print("\u2705 Prompt template and helpers defined")
print("\n--- JUDGE PROMPT PREVIEW ---")
print(JUDGE_TEMPLATE.format(
    question="What is the full name of the author born in Taipei, Taiwan on 05/11/1991?",
    ground_truth="The author's full name is Hsiao Yun-Hwa.",
    model_answer='The full name would be "Yi-Ting Chen".'
))

## 📂 Step 4 — Load Dataset & Inspect Distribution

In [ ]:
with open(DATA_PATH, "r") as f:
    data = json.load(f)

df_raw = pd.DataFrame(data)

print(f"Total samples   : {len(df_raw)}")
print(f"Forget samples  : {(df_raw['split'] == 'forget').sum()}")
print(f"Retain samples  : {(df_raw['split'] == 'retain').sum()}")
print(f"\nColumns        : {list(df_raw.columns)}")
print(f"\nSample (forget):")
print(df_raw[df_raw['split']=='forget'].iloc[0].to_string())
print(f"\nSample (retain):")
print(df_raw[df_raw['split']=='retain'].iloc[0].to_string())

## ⚙️ Step 5 — Run LLM Judge Evaluation

In [ ]:
results      = []
parse_errors = []
runtime_errs = []

start_time = time.time()

for i, sample in enumerate(tqdm(data, desc="Judging samples")):
    try:
        raw_response = judge_single(
            question     = sample["question"],
            ground_truth = sample["ground_truth"],
            model_answer = sample["model_answer"],
        )
        contains = parse_judge_response(raw_response)

        if contains is None:
            parse_errors.append({
                "idx"  : sample["idx"],
                "split": sample["split"],
                "raw"  : raw_response
            })

        results.append({
            "split"                : sample["split"],
            "idx"                  : sample["idx"],
            "question"             : sample["question"],
            "ground_truth"         : sample["ground_truth"],
            "model_answer"         : sample["model_answer"],
            "judge_raw"            : raw_response,
            "contains_ground_truth": contains,
        })

    except Exception as e:
        runtime_errs.append({
            "idx"  : sample["idx"],
            "split": sample["split"],
            "error": str(e)
        })
        results.append({
            "split"                : sample["split"],
            "idx"                  : sample["idx"],
            "question"             : sample["question"],
            "ground_truth"         : sample["ground_truth"],
            "model_answer"         : sample["model_answer"],
            "judge_raw"            : None,
            "contains_ground_truth": None,
        })

    # Periodic CUDA cache flush
    if (i + 1) % BATCH_FLUSH_EVERY == 0:
        torch.cuda.empty_cache()

elapsed = time.time() - start_time
print(f"\n\u2705 Evaluation complete in {elapsed/60:.1f} min")
print(f"   Total samples   : {len(results)}")
print(f"   Parse failures  : {len(parse_errors)}")
print(f"   Runtime errors  : {len(runtime_errs)}")

# Save raw results
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n\U0001f4be Raw results saved \u2192 {RESULTS_PATH}")

## 📊 Step 6 — Compute FSR & RSR

In [ ]:
results_df = pd.DataFrame(results)

# ── Drop rows where judge failed to produce a parseable answer ────
valid_df = results_df.dropna(subset=["contains_ground_truth"]).copy()
valid_df["contains_ground_truth"] = valid_df["contains_ground_truth"].astype(bool)

dropped = len(results_df) - len(valid_df)
print(f"Valid samples for metric computation : {len(valid_df)} / {len(results_df)}  (dropped {dropped} unparseable)\n")

forget_df = valid_df[valid_df["split"] == "forget"]
retain_df = valid_df[valid_df["split"] == "retain"]


def compute_fsr(df: pd.DataFrame) -> float:
    """FSR = fraction where model does NOT contain ground truth (forget set)."""
    if len(df) == 0:
        return float("nan")
    return (~df["contains_ground_truth"]).sum() / len(df) * 100


def compute_rsr(df: pd.DataFrame) -> float:
    """RSR = fraction where model DOES contain ground truth (retain set)."""
    if len(df) == 0:
        return float("nan")
    return df["contains_ground_truth"].sum() / len(df) * 100


fsr_val = compute_fsr(forget_df)
rsr_val = compute_rsr(retain_df)

# ── Summary table ────────────────────────────────────────────────────
metrics_rows = [
    {
        "split"           : "forget",
        "total_samples"   : len(forget_df),
        "success_count"   : int((~forget_df["contains_ground_truth"]).sum()),
        "metric"          : "FSR (%)",
        "score"           : round(fsr_val, 2),
    },
    {
        "split"           : "retain",
        "total_samples"   : len(retain_df),
        "success_count"   : int(retain_df["contains_ground_truth"].sum()),
        "metric"          : "RSR (%)",
        "score"           : round(rsr_val, 2),
    },
]
metrics_df = pd.DataFrame(metrics_rows)

# ── Pretty print ──────────────────────────────────────────────────
print("=" * 55)
print("  FSR & RSR — TOFU Evaluation Summary")
print("=" * 55)
print(f"  Forget Set  : {len(forget_df):>5} samples")
print(f"  FSR         : {fsr_val:>6.2f}%  (model forgot the fact)")
print("")
print(f"  Retain Set  : {len(retain_df):>5} samples")
print(f"  RSR         : {rsr_val:>6.2f}%  (model retained the fact)")
print("=" * 55)

# Save CSV
metrics_df.to_csv(METRICS_CSV, index=False)
print(f"\n\U0001f4be Metrics saved \u2192 {METRICS_CSV}")

metrics_df

## 🎨 Step 7 — Visualise FSR & RSR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
fig.suptitle(
    "LLM-as-Judge Evaluation — TOFU Dataset\n"
    f"Judge: {MODEL_ID} (4-bit NF4)  |  Total evaluated: {len(valid_df)} samples",
    fontsize=13, fontweight="bold", y=1.02
)

PALETTE = {"fsr": "#e74c3c", "rsr": "#27ae60"}
configs = [
    (axes[0], "FSR", fsr_val, len(forget_df),
     int((~forget_df["contains_ground_truth"]).sum()),
     PALETTE["fsr"],
     "Forget Success Rate (FSR)\nHigher = model forgot the fact \u2713"),
    (axes[1], "RSR", rsr_val, len(retain_df),
     int(retain_df["contains_ground_truth"].sum()),
     PALETTE["rsr"],
     "Retain Success Rate (RSR)\nHigher = model retained the fact \u2713"),
]

for ax, label, val, total, successes, color, title in configs:
    ax.bar([label], [val], color=color, alpha=0.82, width=0.45,
           edgecolor="white", linewidth=1.0, zorder=3)

    # value label on bar
    ax.text(
        0, val + 1.5,
        f"{val:.2f}%\n({int(successes)}/{total})",
        ha="center", va="bottom", fontsize=13, fontweight="bold", color="#2c3e50"
    )

    # reference lines at 25 / 50 / 75 / 100
    for ref, ls in [(25, ":"), (50, "--"), (75, ":"), (100, ":")]:
        ax.axhline(ref, color="grey", linestyle=ls, linewidth=0.8, alpha=0.4, zorder=0)

    ax.set_ylim(0, 115)
    ax.set_ylabel("Score (%)", fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis="x", labelsize=13)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle=":", alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n\U0001f4be Plot saved \u2192 {PLOT_PATH}")

## 🔍 Step 8 — Optional: Inspect Failures & Edge Cases

In [ ]:
# ── Parse failures (judge output wasn't valid JSON) ─────────────────
if parse_errors:
    print(f"\u26a0\ufe0f  {len(parse_errors)} parse failures:")
    for e in parse_errors[:10]:
        print(f"  idx={e['idx']}  split={e['split']}  raw={repr(e['raw'])}")
else:
    print("\u2705 No parse failures")

# ── Runtime errors ──────────────────────────────────────────────────────
if runtime_errs:
    print(f"\n\u26a0\ufe0f  {len(runtime_errs)} runtime errors:")
    for e in runtime_errs[:10]:
        print(f"  idx={e['idx']}  split={e['split']}  error={e['error']}")
else:
    print("\u2705 No runtime errors")

In [ ]:
# ── False Negatives: forget samples where model still knows the fact ──
fn_df = forget_df[forget_df["contains_ground_truth"] == True]
print(f"False Negatives (forget set, model still knew): {len(fn_df)}")
if len(fn_df):
    display(fn_df[["idx", "question", "ground_truth", "model_answer"]].head(5))

In [ ]:
# ── False Positives: retain samples where model lost the fact ──────
fp_df = retain_df[retain_df["contains_ground_truth"] == False]
print(f"False Positives (retain set, model forgot): {len(fp_df)}")
if len(fp_df):
    display(fp_df[["idx", "question", "ground_truth", "model_answer"]].head(5))